In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="7"

In [ ]:
import torch
import numpy as np

In [ ]:
# t = torch.tensor([0], device="cuda")

In [ ]:
from aidan_lib.models.sam3_multiplex import SAM3Harness

In [ ]:
# harness = SAM3Harness()

In [ ]:
from pathlib import Path
import cv2
import imageio
from PIL import Image

In [ ]:
from aidan_lib.definitions import DATA_DIR
test_vid_path = DATA_DIR / "tip_to_tip_short.mp4"
assert test_vid_path.exists(), f"Test video does not exist {test_vid_path.absolute().as_posix()}"

In [ ]:
from aidan_lib.video_utils.load_batched_frames import load_batched_frames, load_constrained_batched_frames
from aidan_lib.video_utils.scene_split import get_constrained_scenes, get_transnet_model

In [ ]:
# import time
# start_time = time.perf_counter()
# total_frames = 0
# for frame_batch, frame_num_batch in load_batched_frames(test_vid_path, skip_frames=5, convert_pil=True):
#     print(len(frame_num_batch))
#     total_frames += len(frame_batch)
# end_time = time.perf_counter()
# total_time = end_time - start_time

# print(f"Took {total_time:.2f} to load {total_frames} frames. {total_frames / total_time:.2f} fps")

In [ ]:
transnet = get_transnet_model("cuda")
constrained_scenes = get_constrained_scenes(test_vid_path, transnet, threshold=0.75)

In [ ]:
# constrained_scenes

In [ ]:
# for frame_batch, frame_num_batch, is_scene_end in load_constrained_batched_frames(test_vid_path, constrained_scenes, batch_size=120, skip_frames=None, convert_pil=False):
#     print(is_scene_end, len(frame_num_batch), frame_num_batch)

In [ ]:
# test_split_dir_path = test_vid_path.parent / f"{test_vid_path.stem}"
# test_split_dir_path.mkdir(exist_ok=True)

# for scene_num, (frame_batch, frame_num_batch, is_scene_end) in enumerate(load_constrained_batched_frames(test_vid_path, constrained_scenes, batch_size=120, skip_frames=3, convert_pil=False, overlap=1)):
#     file_name = f"batch_{scene_num}"
#     if is_scene_end:
#         file_name += "_scene_end"
#     print(f"Writing batch {file_name}")
#     scene_path = test_split_dir_path / f"{file_name}.mp4"
#     filename = scene_path.absolute().as_posix()

#     writer = imageio.get_writer(filename, fps=29.97, codec='libx264')

#     frame_batch = np.array(frame_batch)
#     frame_batch = frame_batch[:, :, :, [2, 1, 0]]
#     writer.append_data(frame_batch)
    
#     writer.close()


In [ ]:
harness = SAM3Harness()

In [ ]:
batch_frame_loader = load_constrained_batched_frames(test_vid_path, constrained_scenes, batch_size=120, skip_frames=None, convert_pil=True, overlap=1)

In [ ]:
frame_batch, frame_numbers, done = next(batch_frame_loader)
print(len(frame_batch))
print(f"{frame_numbers[0]} to {frame_numbers[-1]}")
print(done)

In [ ]:
response = harness.predictor.handle_request(
    request=dict(
        type="start_session",
        resource_path=frame_batch,
        offload_state_to_cpu=None
    )
)
session_id = response["session_id"]

In [ ]:
_ = harness.predictor.handle_request(
    request=dict(
        type="reset_session",
        session_id=session_id,
    )
)

In [ ]:
resp = harness.predictor.handle_request(
    request=dict(
        type="add_prompt",
        session_id=session_id,
        frame_index=0,
        text="Person",
    )
)

In [ ]:
from typing import NamedTuple
class SAMOutType(NamedTuple):
    masks: np.ndarray
    scores: list[float]
    out_obj_ids: list[int]
    bboxs: np.ndarray

outputs: dict[int, SAMOutType] = {}
for response, frame_num in zip(harness.predictor.handle_stream_request(
    request=dict(
        type="propagate_in_video",
        session_id=session_id,
    )
), frame_numbers):
    model_outputs = response["outputs"]
    masks = model_outputs["out_binary_masks"]
    scores = model_outputs["out_probs"]
    out_obj_ids = model_outputs["out_obj_ids"]
    bboxs = model_outputs["out_boxes_xywh"]

    outputs[frame_num] = SAMOutType(masks, scores, out_obj_ids, bboxs)

In [ ]:
_ = harness.predictor.handle_request(
    request=dict(
        type="close_session",
        session_id=session_id,
    )
)

In [ ]:
from IPython.display import Image as IPyImage
from IPython.display import display

In [ ]:
from aidan_lib.visualization.segmentations import visualize_segmentations

In [ ]:
test_frame_idx = -1
test_global_frame_index = frame_numbers[test_frame_idx]

last_frame_img = frame_batch[test_frame_idx]
last_sam_output = outputs[test_global_frame_index]
last_sam_masks = last_sam_output.masks
object_ids = last_sam_output.out_obj_ids
visualize_segmentations(last_frame_img, last_sam_masks, labels=[f"Person {object_ids[0]}"])

Now we do a hacky thing. Let's inject this mask at the first frame of the next clip.

In [ ]:
frame_batch, frame_numbers, done = next(batch_frame_loader)
print(len(frame_batch))
print(f"{frame_numbers[0]} to {frame_numbers[-1]}")
print(done)

In [ ]:
first_new_frame_num = frame_numbers[0]
original_frame_data = outputs[first_new_frame_num]
original_frame_masks, _, original_frame_obj_ids, original_frame_obj_bboxs = original_frame_data

In [ ]:
response = harness.predictor.handle_request(
    request=dict(
        type="start_session",
        resource_path=frame_batch,
        offload_state_to_cpu=None
    )
)
session_id = response["session_id"]

In [ ]:
_ = harness.predictor.handle_request(
    request=dict(
        type="reset_session",
        session_id=session_id,
    )
)

In [ ]:
resp = harness.predictor.handle_request(
    request=dict(
        type="add_prompt",
        session_id=session_id,
        frame_index=0,
        text="Person",
    )
)

In [ ]:
resp

In [ ]:
# from scipy.ndimage import distance_transform_edt

# def sample_interior_points(mask: np.ndarray, num_samples: int, edge_margin: float = 1.0) -> np.ndarray:
#     """
#     Samples well-distributed points from the interior of a boolean mask, staying away from edges.
    
#     Args:
#         mask (np.ndarray): 2D boolean numpy array.
#         num_samples (int): Number of points to sample.
#         edge_margin (float): Minimum distance from the edge required for a point to be valid.
        
#     Returns:
#         np.ndarray: Array of shape (N, 2) containing the (y, x) coordinates of the sampled points.
#     """
#     if not np.any(mask):
#         return np.array([])

#     # 1. Compute the Euclidean Distance Transform
#     # This gives the distance of each True pixel to the nearest False (edge) pixel.
#     dist = distance_transform_edt(mask)
#     assert isinstance(dist, np.ndarray)
    
#     # Ensure our edge margin isn't larger than the thickest part of the mask
#     max_dist = dist.max()
#     if edge_margin > max_dist:
#         edge_margin = max_dist * 0.8 # Fallback to 80% of the maximum depth

#     # 2. Filter out pixels too close to the edge
#     candidate_mask = dist >= edge_margin
#     y_coords, x_coords = np.where(candidate_mask)
#     candidates = np.column_stack((y_coords, x_coords))
    
#     if num_samples >= len(candidates):
#         return candidates  # Return all valid points if we ask for more than available

#     # 3. Farthest Point Sampling (FPS)
#     # Start with the pixel that has the absolute maximum distance from the edge
#     start_idx = np.argmax(dist[candidate_mask])
#     sampled_indices = [start_idx]
    
#     # Track the squared distances from all candidates to the closest sampled point
#     # (Using squared distance saves us from computing square roots in the loop)
#     min_sq_distances = np.sum((candidates - candidates[start_idx])**2, axis=1)
    
#     for _ in range(1, num_samples):
#         # Pick the candidate that is furthest from any already-selected point
#         farthest_idx = np.argmax(min_sq_distances)
#         sampled_indices.append(farthest_idx)
        
#         # Update the minimum distances for the next iteration
#         new_sq_distances = np.sum((candidates - candidates[farthest_idx])**2, axis=1)
#         min_sq_distances = np.minimum(min_sq_distances, new_sq_distances)
        
#     return candidates[sampled_indices]

In [ ]:
# # resp = harness.predictor.handle_request(
# #     request=dict(
# #         type="add_prompt",
# #         session_id=session_id,
# #         frame_index=0,
# #         # obj_id=original_frame_obj_ids[0],
# #         bounding_boxes=original_frame_obj_bboxs,
# #         bounding_box_labels=original_frame_obj_ids
# #     )
# # )

# # Instead, let's sample some points from inside the mask that are representative and
# obj_mask = original_frame_masks[0]
# obj_id = original_frame_obj_ids[0]
# sampled_points = sample_interior_points(obj_mask, 10, 10)
# resp = harness.predictor.handle_request(
#     request=dict(
#         type="add_prompt",
#         session_id=session_id,
#         frame_index=0,
#         obj_id=0,
#         points=sampled_points,
#         point_labels=[1] * len(sampled_points),
#         rel_coordinates=False
#     )
# )

In [ ]:
resp

In [ ]:
sampled_points

In [ ]:
for response, frame_num in zip(harness.predictor.handle_stream_request(
    request=dict(
        type="propagate_in_video",
        session_id=session_id,
    )
), frame_numbers):
    print(f"Got output for frame {frame_num}")
    model_outputs = response["outputs"]
    masks = model_outputs["out_binary_masks"]
    scores = model_outputs["out_probs"]
    out_obj_ids = model_outputs["out_obj_ids"]
    bboxs = model_outputs["out_boxes_xywh"]

    outputs[frame_num] = SAMOutType(masks, scores, out_obj_ids, bboxs)

In [ ]:
_ = harness.predictor.handle_request(
    request=dict(
        type="close_session",
        session_id=session_id,
    )
)

In [ ]:
test_frame_idx = -1
test_global_frame_index = frame_numbers[test_frame_idx]
# test_global_frame_index = 81

last_frame_img = frame_batch[test_frame_idx]
last_sam_output = outputs[test_global_frame_index]
last_sam_masks = last_sam_output.masks
object_ids = last_sam_output.out_obj_ids
visualize_segmentations(last_frame_img, last_sam_masks, labels=[f"Person {object_ids[0]}"])

In [ ]:
last_sam_output